# Phase 3 — Custom Gymnasium Environment (v2)

**What changed from v1:**
The environment now takes two DataFrames:
- `df` → normalised (scaled) features, used only for observations fed to the agent
- `raw_df` → un-scaled OHLCV with real dollar prices, used only for trade execution

This fixes the negative price / negative shares bug caused by RobustScaler producing negative Close values.

---
## Cell 1 — Imports & path setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from env.trading_env import TradingEnv
print('[PASS] TradingEnv v2 imported successfully.')

---
## Cell 2 — Load BOTH scaled and raw DataFrames

This is the key change. We load:
- `train_df` / `val_df` / `test_df` — normalised, from `data/processed/`
- `train_raw` / `val_raw` / `test_raw` — raw dollar prices, from `data/raw/`

We then align them by index so rows match exactly.

In [ ]:
TICKER = 'AAPL'

# --- Scaled features (from Phase 2 pipeline) ---
train_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train.csv'),
                       index_col=0, parse_dates=True)
val_df   = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'val.csv'),
                       index_col=0, parse_dates=True)
test_df  = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'test.csv'),
                       index_col=0, parse_dates=True)

# --- Raw prices (un-scaled, real dollars) ---
raw_all = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'raw', f'{TICKER}.csv'),
                      index_col=0, parse_dates=True)

# Flatten multi-level columns if present
if isinstance(raw_all.columns, pd.MultiIndex):
    raw_all.columns = raw_all.columns.get_level_values(0)

# Align raw splits to the same index as the scaled splits
train_raw = raw_all.loc[train_df.index]
val_raw   = raw_all.loc[val_df.index]
test_raw  = raw_all.loc[test_df.index]

# Sanity checks
assert len(train_df) == len(train_raw), 'FAIL: train_df and train_raw row count mismatch'
assert len(val_df)   == len(val_raw),   'FAIL: val_df and val_raw row count mismatch'
assert len(test_df)  == len(test_raw),  'FAIL: test_df and test_raw row count mismatch'
assert (train_raw['Close'] > 0).all(),  'FAIL: Negative prices in train_raw'

print(f'Scaled train  : {train_df.shape}   Close range: {train_df["Close"].min():.2f} to {train_df["Close"].max():.2f}  (normalised — can be negative)')
print(f'Raw train     : {train_raw.shape}  Close range: ${train_raw["Close"].min():.2f} to ${train_raw["Close"].max():.2f}  (real dollars)')
print()
print('[PASS] Both DataFrames loaded and aligned.')

---
## Cell 3 — Instantiate environment with both DataFrames

In [ ]:
WINDOW_SIZE = 10
N_FEATURES  = train_df.shape[1]

env = TradingEnv(
    df               = train_df,      # normalised features → observations
    raw_df           = train_raw,     # real dollar prices  → trade execution
    initial_balance  = 10_000.0,
    transaction_cost = 0.001,
    window_size      = WINDOW_SIZE,
)

print('Observation space:', env.observation_space)
print('Action space     :', env.action_space)
print('n_features       :', env.n_features)

assert env.observation_space.shape == (WINDOW_SIZE, N_FEATURES)
assert env.action_space.n == 3
print()
print('[PASS] Environment instantiated with correct spaces.')

---
## Cell 4 — Test reset()

In [ ]:
obs, info = env.reset()

print(f'obs shape  : {obs.shape}')
print(f'obs dtype  : {obs.dtype}')
print(f'balance    : ${env.balance:,.2f}')
print(f'shares     : {env.shares}')
print(f'step       : {env.current_step}')
print(f'first real price at step {env.current_step}: ${env._current_price():.2f}')
print()

assert obs.shape  == (WINDOW_SIZE, N_FEATURES)
assert obs.dtype  == np.float32
assert env.balance == 10_000.0
assert env.shares  == 0.0
assert env._current_price() > 0,  'FAIL: Price is still negative — check raw_df is loaded correctly'

print('[PASS] reset() correct. Price is a real positive dollar value.')

---
## Cell 5 — Test each action (the cells that failed before)

In [ ]:
# ---- HOLD ----
obs, _ = env.reset()
balance_before = env.balance
obs2, reward, *_ = env.step(0)
assert env.balance == balance_before
assert env.shares  == 0.0
print(f'HOLD  | reward={reward:.6f}  balance=${env.balance:,.2f}  shares={env.shares:.4f}')
print('[PASS] Hold')

# ---- BUY ----
obs, _ = env.reset()
price  = env._current_price()
obs2, reward, *_ = env.step(1)
print(f'BUY   | reward={reward:.6f}  balance=${env.balance:,.2f}  shares={env.shares:.4f}  price=${price:.2f}')
assert env.balance < 10_000.0,   'FAIL: Balance should decrease after Buy'
assert env.shares  > 0.0,        'FAIL: Shares should be positive after Buy'
assert price       > 0.0,        f'FAIL: Price is {price:.4f} — should be a real dollar value'
print('[PASS] Buy — real positive price, correct balance deduction, positive shares')

# ---- SELL ----
env.step(1)   # buy again to have shares
shares_before = env.shares
bal_before    = env.balance
obs2, reward, *_ = env.step(2)
print(f'SELL  | reward={reward:.6f}  balance=${env.balance:,.2f}  shares={env.shares:.4f}')
assert env.shares  == 0.0,         'FAIL: Shares should be 0 after Sell'
assert env.balance > bal_before,   'FAIL: Balance should increase after Sell'
print('[PASS] Sell — shares zeroed, balance increased')

# ---- SELL with no shares ----
obs, _ = env.reset()
bal = env.balance
env.step(2)
assert env.balance == bal and env.shares == 0.0
print('[PASS] Sell with no shares — no-op confirmed')

---
## Cell 6 — Transaction cost verification

In [ ]:
env_no_cost   = TradingEnv(train_df, train_raw, transaction_cost=0.0)
env_with_cost = TradingEnv(train_df, train_raw, transaction_cost=0.001)

for e in [env_no_cost, env_with_cost]:
    e.reset()
    e.step(1)   # buy
    e.step(2)   # sell

print(f'No cost   — balance: ${env_no_cost.balance:,.4f}')
print(f'With cost — balance: ${env_with_cost.balance:,.4f}')
print(f'Cost drag : ${env_no_cost.balance - env_with_cost.balance:.4f}')

assert env_with_cost.balance < env_no_cost.balance
assert env_with_cost.balance < 10_000.0
print('[PASS] Transaction costs applied correctly.')

---
## Cell 7 — Gymnasium check_env()

In [ ]:
from gymnasium.utils.env_checker import check_env

test_env = TradingEnv(train_df, train_raw)
print('Running check_env()...')
check_env(test_env, warn=True)
print()
print('[PASS] check_env() passed.')

---
## Cell 8 — Full random episode

In [ ]:
env_ep = TradingEnv(train_df, train_raw)
obs, _ = env_ep.reset()

total_reward = 0
steps = 0
done  = False

while not done:
    action = env_ep.action_space.sample()
    obs, reward, terminated, truncated, info = env_ep.step(action)
    total_reward += reward
    steps += 1
    done = terminated or truncated

final_value = info['portfolio_value']
pct_return  = (final_value - 10_000) / 10_000 * 100
n_trades    = len(env_ep.trade_log)

print(f'Steps           : {steps}')
print(f'Trades executed : {n_trades}')
print(f'Final portfolio : ${final_value:,.2f}')
print(f'Return          : {pct_return:+.2f}%')

assert steps > 0
assert final_value > 0
print()
print('[PASS] Full episode completed without errors.')

---
## Cell 9 — Portfolio vs Buy & Hold chart

In [ ]:
portfolio_series = env_ep.get_portfolio_series()
trade_log        = env_ep.get_trade_log()

start_price = train_raw['Close'].iloc[WINDOW_SIZE]
bnh_series  = 10_000 * (train_raw['Close'].iloc[WINDOW_SIZE:] / start_price)
bnh_return  = (train_raw['Close'].iloc[-1] / start_price - 1) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(portfolio_series.values,
             label='Random agent', color='steelblue', linewidth=1)
axes[0].plot(bnh_series.values[:len(portfolio_series)],
             label='Buy & Hold',   color='orange',    linewidth=1, linestyle='--')
axes[0].axhline(10_000, color='gray', linestyle=':', linewidth=0.8)
axes[0].set_title('Portfolio value — random agent vs Buy & Hold')
axes[0].set_ylabel('Portfolio ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_raw['Close'].iloc[WINDOW_SIZE:].values,
             color='steelblue', linewidth=0.8, label='Close price')

if len(trade_log) > 0:
    buys  = trade_log[trade_log['action'] == 'BUY']
    sells = trade_log[trade_log['action'] == 'SELL']
    buy_idx  = (buys['step'].values  - WINDOW_SIZE).clip(0, len(train_raw) - WINDOW_SIZE - 1)
    sell_idx = (sells['step'].values - WINDOW_SIZE).clip(0, len(train_raw) - WINDOW_SIZE - 1)
    axes[1].scatter(buy_idx,  train_raw['Close'].iloc[WINDOW_SIZE:].values[buy_idx],
                    marker='^', color='green', s=40, zorder=5, label=f'Buy ({len(buys)})')
    axes[1].scatter(sell_idx, train_raw['Close'].iloc[WINDOW_SIZE:].values[sell_idx],
                    marker='v', color='red',   s=40, zorder=5, label=f'Sell ({len(sells)})')

axes[1].set_title('Price chart with trade signals (real dollar prices)')
axes[1].set_ylabel('Price ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Buy & Hold return : {bnh_return:+.2f}%')
print(f'Random agent      : {pct_return:+.2f}%')
print()
print('[INFO] Random agent underperforming is expected. Phase 4 trains the real agent.')

---
## Cell 10 — DummyVecEnv compatibility check

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv

vec_env = DummyVecEnv([lambda: TradingEnv(train_df, train_raw)])

obs = vec_env.reset()
print(f'VecEnv obs shape : {obs.shape}   (should be (1, {WINDOW_SIZE}, {N_FEATURES}))')

obs, rewards, dones, infos = vec_env.step([1])
print(f'After step(Buy)  : reward={rewards[0]:.6f}  portfolio=${infos[0]["portfolio_value"]:,.2f}')

assert obs.shape == (1, WINDOW_SIZE, N_FEATURES)
assert infos[0]['price'] > 0, 'FAIL: Price still negative in VecEnv'
print()
print('[PASS] DummyVecEnv wraps TradingEnv v2 correctly. Ready for Phase 4.')

---
## Cell 11 — Phase 3 summary

In [ ]:
print('=' * 55)
print('  Phase 3 v2 — all checks')
print('=' * 55)
checks = [
    'TradingEnv v2 imported',
    'Scaled df and raw_df both loaded and index-aligned',
    'Raw Close prices are real positive dollar values',
    'Observation space uses normalised features',
    'Trade execution uses real dollar prices from raw_df',
    'Hold — no state change',
    'Buy  — correct balance deduction, positive shares',
    'Sell — all shares liquidated, balance increased',
    'Sell with 0 shares — no-op',
    'Transaction cost reduces balance vs no-cost env',
    'check_env() passed',
    'Full episode ran without error',
    'DummyVecEnv wraps correctly for SB3',
]
for c in checks:
    print(f'  [PASS]  {c}')
print()
print('==> Phase 3 complete. Move to Phase 4: DQN and PPO agent training.')